# Shadowseed Layer G — gratis 14B-proef

Dit notebook gebruikt **Qwen2.5-14B-Instruct** met gratis NVIDIA GPU-rekentijd (Kaggle of Colab) en open modelgewichten. Geen betaalde Hugging Face Jobs, creditcard of commerciële API.

**Vooraf:** kies een gratis **GPU T4 of P100** in Notebook/Runtime Settings. Een 16GB-GPU vereist **NF4-kwantisatie naar 4-bit**. Dat verandert de numerieke modelrepresentatie; dit is een *instrumentpilot*, geen full-precision H-Neurons-replicatie.

Bij geen GPU, te weinig schijf of een quotumweigeringsmelding stopt het notebook. Het valt niet terug op een kleiner model. Alleen de meegeleverde fictieve invoer wordt gebruikt.

## 1. GPU-controle (voor downloads)

In [ ]:
import os, sys, subprocess, shutil, hashlib, json, zipfile
from pathlib import Path
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # ook bij Kaggle 2x T4 bewust één GPU
import torch
assert torch.cuda.is_available(), "Geen gratis NVIDIA GPU toegewezen. Kies T4/P100 in notebookinstellingen."
gpu = torch.cuda.get_device_properties(0)
size = gpu.total_memory / 1024**3
assert torch.cuda.device_count() == 1 and size >= 14 and gpu.major >= 6, "Deze proef vereist één CUDA GPU >=14 GiB met compute >=6.0."
print(f"Gratis GPU: {gpu.name}; {size:.1f} GiB; compute {gpu.major}.{gpu.minor}")

## 2. Open-source software en exacte broncode

In [ ]:
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.1", "accelerate==1.11.0", "bitsandbytes==0.48.2",
    "huggingface_hub==0.36.0", "safetensors>=0.5"])
work = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content')
work.mkdir(parents=True, exist_ok=True)
repo = work / 'shadowseed-pro-free'
if not repo.exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', 'feat/layer-g-vnext-preflight',
        'https://github.com/E-AI-MODEL/shadowseed-pro.git', str(repo)])
source_sha = subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()
assert len(source_sha)==40
cache = Path('/kaggle/temp/shadowseed-hf-cache') if Path('/kaggle/temp').exists() else work/'shadowseed-hf-cache'
cache.mkdir(parents=True,exist_ok=True)
assert shutil.disk_usage(cache).free/1024**3 >= 34, '>=34 GiB vrije schijf vereist; geen automatische modeldowngrade.'
os.environ['HF_HOME'] = str(cache)
sys.path.insert(0,str(repo/'research/src'))
print('Instrumentbron:', source_sha, 'Cache:', cache)

## 3. Laad het echte 14B-model in 4-bit en meet activaties tijdens generatie

In [ ]:
from huggingface_hub import model_info
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from shadowseed_research.benchmark.layer_g_live_generation import (
    GenerationRequest, HFOriginalGenerationReader, write_generation_observation)
import importlib.metadata as metadata
MODEL = 'Qwen/Qwen2.5-14B-Instruct'
model_sha = model_info(MODEL).sha
assert len(model_sha) == 40, 'Modelcommit niet vast te pinnen.'
fictional = (repo/'research/fixtures/layer_g_vnext_synthetic_prompt.txt').read_text(encoding='utf-8')
assert fictional.strip() and 'synthetic' in 'layer_g_vnext_synthetic_prompt.txt'
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=model_sha, trust_remote_code=False)
formatted = tokenizer.apply_chat_template([{'role':'user','content':fictional}],
    tokenize=False, add_generation_prompt=True)
model = AutoModelForCausalLM.from_pretrained(MODEL, revision=model_sha,
    quantization_config=quant, device_map={'':0}, torch_dtype=torch.float16,
    low_cpu_mem_usage=True, trust_remote_code=False).eval()
assert model.is_loaded_in_4bit and model.device.type == 'cuda', '14B niet volledig als NF4 op GPU geladen.'
reader = HFOriginalGenerationReader(model, tokenizer, model_id=MODEL,
    model_revision=model_sha, layers=(
      'model.layers.12.mlp.down_proj', 'model.layers.36.mlp.down_proj'),
    max_input_tokens=1024, max_new_tokens=16)
record = reader.capture(GenerationRequest.from_original_prompt(formatted))
print('14B gegenereerde tokens:',len(record.token_ids))
print('MLP-activaties:',{k:v.shape for k,v in record.layer_features.items()})

## 4. Vastleggen en controleren — géén kennisclaim

In [ ]:
output = work/'shadowseed-layerg-14b-nf4'
index = 1
while output.exists():
    output = work/f'shadowseed-layerg-14b-nf4-{index}'
    index += 1
runtime = {name: metadata.version(name) for name in ['torch','transformers','bitsandbytes','accelerate','huggingface_hub','numpy']}
runtime['gpu'] = gpu.name
runtime['source_git_sha'] = source_sha
manifest = write_generation_observation(record, output, runtime=runtime)
provenance = {
    'model_id':MODEL,'model_revision':model_sha,'source_git_sha':source_sha,
    'quantization':'NF4 4-bit, double quant, float16 computation',
    'precision_comparison':'NOT full precision; no cross-precision neuron claim',
    'cost_policy':'free_gpu_only; no billing calls',
    'research_claim':'none',
    'manifest_sha256':hashlib.sha256((output/'manifest.json').read_bytes()).hexdigest(),
}
(output/'free_gpu_provenance.json').write_text(json.dumps(provenance,sort_keys=True,indent=2)+'\n')
assert manifest['capture_mode']=='synchronous_original_generation'
assert manifest['research_claim']=='none'
assert manifest['generated_token_count']==len(manifest['forward_token_counts'])
assert manifest['features_sha256']==hashlib.sha256((output/'decision_activations.npz').read_bytes()).hexdigest()
zip_path = output.with_suffix('.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for file in sorted(output.iterdir()): z.write(file,arcname=file.name)
print('MODEL', MODEL, 'SHA', model_sha)
print('Activatieinstrument:',manifest['status'], 'tokens:',manifest['generated_token_count'])
print('Resultaat:',zip_path)
from IPython.display import display, FileLink
display(FileLink(str(zip_path)))

## Wat dit nog niet bewijst

Dit notebook levert, bij geslaagde uitvoering, **echte activatiemetingen** van een 14B-model in **4-bit**. Het laat niet zien dat we verborgen kennis of een niet-uitgesproken relatie hebben gevonden. Daarvoor zijn onafhankelijke beoordelingen, geschikte controles, preregistratie en replicatie nodig.

Het ZIP-bestand is afkomstig van fictieve input en kan worden toegevoegd aan [issue #203](https://github.com/E-AI-MODEL/shadowseed-pro/issues/203). Gratis GPU-uren zijn beperkt; het notebook start nooit betaalde cloudjobs.